In [ ]:
import georinex as gr
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import gnss_tools

In [ ]:
dcb = gnss_tools.processDCB(r"data/CAS0OPSRAP_20260810000_01D_01D_DCB.BIA.gz")

In [ ]:
dcb

In [ ]:
# Constants
c = 299792458.0 # m/s
f1 = 1.57542e9  # Hz
f2 = 1.22760e9  # Hz
f5 = 1.17645e9  # Hz


In [ ]:
file = r"data/ABMF00GLP_R_20260810000_01D_30S_MO.crx.gz"
obs = gr.load(file, use=['G','E'], fast=True)

In [ ]:
obs

Use our function from before

In [ ]:


obs['L1C'] = c * obs['L1C'] / f1
obs['L2W'] = c * obs['L2W'] / f2

# proportionality factor
tecScale = ((f2**2) * (f1**2) / (40.3 * (f1**2 - f2**2))) * 1e-16 # convert to TECU

# phase tec
TEC_phase = tecScale * (obs['L1C'] - obs['L2W'])

# Code TEC
TEC_code = tecScale * (obs['C2W'] - obs['C1C'])

In [ ]:
data = gnss_tools.simple_phase_level(TEC_code, TEC_phase, max_error=4.5)

Plot our TEC!

In [ ]:
plt.plot(data['sTEC'])

Check if our station is in the IGS dataset

In [ ]:
'ABMF' in dcb['stn']

In [ ]:
dcb_abmf = dcb.sel(stn='ABMF')

In [ ]:
dcb_abmf

What is receiver bias?

In [ ]:
dcb_abmf['stnDCB'].sel(combination='C1C_C2W')

Just fix receiver biases

In [ ]:
plt.plot(data['sTEC'] + dcb_abmf['stnDCB'].sel(combination='C1C_C2W').data*1e-16)

Just fix satellite biases

In [ ]:
plt.plot(data['sTEC'] + dcb_abmf.satDCB.sel(combination='C1C_C2W').isel(date=0)*1e-16)

In [ ]:
plt.plot(data['sTEC'] + dcb_abmf['stnDCB'].sel(combination='C1C_C2W').data*1e-16 + dcb_abmf.satDCB.sel(combination='C1C_C2W').isel(date=0)*1e-16)

If your station isn't in the IGS network you have to either level to an IONEX GIM:
IONEX standard http://ftp.aiub.unibe.ch/ionex/draft/ionex11.pdf
https://link.springer.com/article/10.1007/s10291-012-0284-6
https://agupubs.onlinelibrary.wiley.com/doi/full/10.1002/2015JA021639
https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2023SW003611?af=R

Or use something like Minimization of Standard Deviation: (works well! but needs a whole day of data!!)
Ma and Maruyama 2003 https://angeo.copernicus.org/articles/21/2083/2003/